In [1]:
import json
from kafka import KafkaProducer, KafkaConsumer

In [2]:
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(bootstrap_servers = [server],
                        value_serializer = json_serializer)

producer.bootstrap_connected()

True

In [3]:
import pandas as pd

df = pd.read_csv('green_tripdata_2019-10.csv')
data = df.iloc[:,[1,2,5,6,7,8,9]]
# data['lpep_pickup_datetime'] = pd.to_datetime(data['lpep_pickup_datetime'])
# data['lpep_dropoff_datetime']= pd.to_datetime(data['lpep_dropoff_datetime'])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476386 entries, 0 to 476385
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   lpep_pickup_datetime   476386 non-null  object 
 1   lpep_dropoff_datetime  476386 non-null  object 
 2   PULocationID           476386 non-null  int64  
 3   DOLocationID           476386 non-null  int64  
 4   passenger_count        387007 non-null  float64
 5   trip_distance          476386 non-null  float64
 6   fare_amount            476386 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 25.4+ MB


/Users/23491378/opt/anaconda3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
from time import time
topic_name = 'green-trips'

t0 = time()

for _,row in data.iterrows():
    producer.send(topic_name, value=row.to_dict())
producer.flush()
t1 = time()
took = t1-t0

In [5]:
print(took)

46.41011309623718


In [6]:
consumer = KafkaConsumer(topic_name, bootstrap_servers=[server], value_deserializer=lambda v: json.loads(v.decode('utf-8')))
print(consumer.bootstrap_connected())

# for message in consumer:
#     print(message.value)
#     break #only check the first message.
# # consumer.close()
#     print ("%s:%d:%d: key=%s value=%s" % (message.topic, message.partition,
#                                               message.offset, message.key,
#                                               message.value))

True
